In [36]:
#  IMPORT LIBRARIES

import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.sparse import hstack
from torch.utils.data import Dataset, DataLoader




# SAFE DATA LOADING


df = pd.read_csv(
    "Phishing_Email.csv",
    engine="python",        # tolerant parser
    on_bad_lines="skip"     # removes corrupted rows safely
)


# CREATE TARGET LABEL


df["label"] = df["Email Type"].map({
    "Safe Email": 0,
    "Phishing Email": 1
}).astype(np.float32)


# BASIC CLEANING


# Remove missing rows
df = df.dropna()

# Ensure text type consistency
df["Email Text"] = df["Email Text"].astype(str)

# Remove HTML first
df["Email Text"] = df["Email Text"].str.replace(r"<.*?>", "", regex=True)

# Normalize whitespace
df["Email Text"] = df["Email Text"].str.replace(r"\s+", " ", regex=True)

# Trim + lowercase normalization
df["Email Text"] = df["Email Text"].str.strip().str.lower()


# FILTERING (AFTER CLEANING)


# Remove very short / non-informative emails
df = df[df["Email Text"].str.len() > 20]


In [28]:
#  FEATURE ENGINEERING (MANUAL FEATURES)

# These features capture structural phishing signals beyond text embeddings
def extract_features(text):
    return {
        # URLs are strong phishing indicators
        "num_links": len(re.findall(r"http[s]?://", text)),

        # Excess punctuation often indicates urgency manipulation
        "num_exclamations": text.count("!"),

        # ALL CAPS words often indicate spam/phishing
        "num_uppercase_words": sum(1 for w in text.split() if w.isupper()),

        # Presence of urgency-related keywords
        "has_urgent_words": int(any(w in text for w in [
            "urgent", "immediately", "verify", "account suspended", "action required"
        ])),

        # Financial keywords often used in scams
        "has_money_words": int(any(w in text for w in [
            "bank", "account", "payment", "invoice"
        ])),

        # very long emails can be suspicious
        "text_length": len(text)
    }

# Apply feature extraction to every email
X_manual = pd.DataFrame([extract_features(t) for t in df["Email Text"]])



# TF-IDF TEXT FEATURES

# TF-IDF converts raw text into numerical representation
# capturing word importance across dataset
tfidf = TfidfVectorizer(
    max_features=10000,   # limit vocabulary size for efficiency
    ngram_range=(1, 3),   # captures phrases like "click here now"
    stop_words="english"
)


In [29]:
# TRAIN / VAL / TEST SPLIT

# We split beofre fitting TF-IDF or scaling to avoid leakage

X_train_text, X_temp_text, X_train_manual, X_temp_manual, y_train, y_temp = train_test_split(
    df["Email Text"],
    X_manual,
    df["label"].values,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

X_val_text, X_test_text, X_val_manual, X_test_manual, y_val, y_test = train_test_split(
    X_temp_text,
    X_temp_manual,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)


# FIT TF-IDF ONLY ON TRAIN DATA to prevent data leakage
# Never fit preprocessing on validation/test data

X_train_text = tfidf.fit_transform(X_train_text)
X_val_text   = tfidf.transform(X_val_text)
X_test_text  = tfidf.transform(X_test_text)



# 7. SCALE MANUAL FEATURES ON TRAIN ONLY

# StandardScaler ensures numerical features are normalized
# Mean/std computed only from training data

scaler = StandardScaler()

X_train_manual = scaler.fit_transform(X_train_manual)
X_val_manual   = scaler.transform(X_val_manual)
X_test_manual  = scaler.transform(X_test_manual)



# COMBINE FEATURES

# We merge sparse TF-IDF + dense manual features into one matrix

X_train = hstack([X_train_text, X_train_manual]).astype(np.float32)
X_val   = hstack([X_val_text, X_val_manual]).astype(np.float32)
X_test  = hstack([X_test_text, X_test_manual]).astype(np.float32)


In [33]:
# 9. PYTORCH DATASET

# Converts sparse matrix into dense tensor for ANN training

class EmailDataset(Dataset):
    def __init__(self, X, y):
        # Convert sparse matrix → dense → tensor
        self.X = torch.tensor(X.toarray(), dtype=torch.float32)

        # Reshape labels to [N, 1] for binary classification
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Create DataLoaders (mini-batches for training stability)
train_loader = DataLoader(EmailDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(EmailDataset(X_val, y_val), batch_size=256)
test_loader  = DataLoader(EmailDataset(X_test, y_test), batch_size=256)



In [34]:
# NEURAL NETWORK MODEL (ANN)

# OUTPUT LAYER DESIGN:
# Binary classification → 1 neuron (logit output)

class ANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # OUTPUT LAYER:
            # 1 neuron → binary classification logit
            nn.Linear(128, 1)
        )

    def forward(self, x):
        return self.net(x)


# TRAINING SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ANN(X_train.shape[1]).to(device)

# LOSS FUNCTION (binary classification standard)
# BCEWithLogitsLoss = sigmoid + BCE combined (stable)
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 12. TRAINING LOOP

epochs = 20

for epoch in range(epochs):


    # TRAINING PHASE

    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)              # raw model output (logits)
        loss = criterion(logits, yb)    # loss computed directly on logits

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # VALIDATION PHASE

    model.eval()
    val_loss = 0
    probs_all = []
    y_all = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item()

            # Convert logits-> probabilities (only for evaluation)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()

            probs_all.extend(probs)
            y_all.extend(yb.cpu().numpy().flatten())

    val_loss /= len(val_loader)


    # METRICS (Binary Classification)

    preds = (np.array(probs_all) > 0.5).astype(int)

    acc = accuracy_score(y_all, preds)
    auc = roc_auc_score(y_all, probs_all)
    f1  = f1_score(y_all, preds)

    # Print every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss  : {val_loss:.4f}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"ROC-AUC   : {auc:.4f}")
        print(f"F1 Score  : {f1:.4f}")
        print("----------------------------")





Epoch 5
Train Loss: 0.0013
Val Loss  : 0.0415
Accuracy  : 0.9904
ROC-AUC   : 0.9988
F1 Score  : 0.9875
----------------------------
Epoch 10
Train Loss: 0.0025
Val Loss  : 0.0721
Accuracy  : 0.9866
ROC-AUC   : 0.9979
F1 Score  : 0.9824
----------------------------
Epoch 15
Train Loss: 0.0008
Val Loss  : 0.0784
Accuracy  : 0.9891
ROC-AUC   : 0.9981
F1 Score  : 0.9858
----------------------------
Epoch 20
Train Loss: 0.0001
Val Loss  : 0.0854
Accuracy  : 0.9886
ROC-AUC   : 0.9986
F1 Score  : 0.9852
----------------------------


In [35]:

# FINAL TEST EVALUATION

model.eval()

probs_all, y_all = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)

        logits = model(xb)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()

        probs_all.extend(probs)
        y_all.extend(yb.numpy().flatten())

test_preds = (np.array(probs_all) > 0.5).astype(int)

print("\n=== FINAL TEST RESULTS ===")
print(f"Accuracy : {accuracy_score(y_all, test_preds):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_all, probs_all):.4f}")
print(f"F1 Score : {f1_score(y_all, test_preds):.4f}")


=== FINAL TEST RESULTS ===
Accuracy : 0.9916
ROC-AUC  : 0.9990
F1 Score : 0.9891
